# Datasets

In [4]:
import pandas as pd
import numpy as np
import re

In [8]:
def salvar_xy_csv(x, y, nome_arquivo="data.csv"):
    df = pd.DataFrame({"x": x, "y": y})
    df.to_csv(nome_arquivo, index=False)

folder = '../data/covid_tests_data/'
filter_func = lambda i: lambda x: float(re.findall('[0-9].[0-9][0-9]', x.replace(',', '.'))[i])

In [9]:
tables = [pd.read_csv(folder+'raw/table_a.csv'),
          pd.read_csv(folder+'raw/table_b.csv'),
          pd.read_csv(folder+'raw/table_c.csv'),
          pd.read_csv(folder+'raw/table_d.csv'),
          pd.read_csv(folder+'raw/table_e.csv'),
          pd.read_csv(folder+'raw/table_pcr_a.csv'),
          pd.read_csv(folder+'raw/table_pcr_b.csv'),
          pd.read_csv(folder+'raw/table_pcr_c.csv'),
         ]

letters = ['a', 'b', 'c', 'd', 'e', 'a_pcr', 'b_pcr', 'c_pcr']
for i in range(8):

    tables[i]['sensitivity']          = tables[i]['Sensitivity [95% CI]'].apply(filter_func(0))
    tables[i]['sensitivity_ci_lower'] = tables[i]['Sensitivity [95% CI]'].apply(filter_func(1))
    tables[i]['sensitivity_ci_upper'] = tables[i]['Sensitivity [95% CI]'].apply(filter_func(2))
    tables[i]['specificity']          = tables[i]['Specificity [95% CI]'].apply(filter_func(0))
    tables[i]['specificity_ci_lower'] = tables[i]['Specificity [95% CI]'].apply(filter_func(1))
    tables[i]['specificity_ci_upper'] = tables[i]['Specificity [95% CI]'].apply(filter_func(2))
    tables[i]['type']                 = letters[i]
    
    del tables[i]['Sensitivity [95% CI]']
    del tables[i]['Specificity [95% CI]']
    
df = pd.concat(tables)
x = df.sensitivity.values
y = df.specificity.values
salvar_xy_csv(x, y, nome_arquivo='../data/treated_data/covid_tests.csv')

In [11]:
df = pd.read_csv('../data/study_lymphangeiography.csv')
x = df.Sensitivity.values/100
y = df.Specificity.values/100
salvar_xy_csv(x, y, nome_arquivo='../data/treated_data/study_lymphangeiography.csv')

In [15]:
df = pd.read_csv('../data/Vaccination_Coverage_and_Exemptions_among_Kindergartners_20250909.csv')
DTP_data = df[(df['Vaccine/Exemption'] == 'DTP, DTaP, or DT')&(df['School Year'] == '2024-25')].sort_values(by='Geography')
DTP_data_23 = df[(df['Vaccine/Exemption'] == 'DTP, DTaP, or DT')&(df['School Year'] == '2021-22')].sort_values(by='Geography')
MMR_data = df[(df['Vaccine/Exemption'] == 'MMR')&(df['School Year'] == '2024-25')].sort_values(by='Geography')
Hepatite_data = df[(df['Vaccine/Exemption'] == 'Hepatitis B')&(df['School Year'] == '2024-25')].sort_values(by='Geography')
dataset = pd.DataFrame(index=DTP_data['Geography'], data={'DTP': DTP_data['Estimate (%)'].values, 
                                                          'MMR': MMR_data['Estimate (%)'].values,
                                                          'DTP23': DTP_data_23['Estimate (%)'].values,
                                                          'Hepatite': Hepatite_data['Estimate (%)'].values})
dataset.drop(['Alabama', 'Illinois', 'Maine', 'South Dakota', ], inplace=True)
dataset['DTP'] = np.float64(dataset['DTP'])/100; dataset['MMR'] = np.float64(dataset['MMR'])/100; 
dataset['DTP23'] = np.float64(dataset['DTP23'])/100; dataset['Hepatite'] = np.float64(dataset['Hepatite'])/100;
dataset.to_csv('../data/treated_data/vaccination_coverage_usa.csv')

In [18]:
df_dtp = pd.read_csv('../data/share-of-children-immunized-dtp3/share-of-children-immunized-dtp3.csv',
                     names=['Entity', 'Code', 'Year', 'Share'], header=0)
df_measles = pd.read_csv('../data/share-of-children-vaccinated-against-measles/share-of-children-vaccinated-against-measles.csv',
                         names=['Entity', 'Code', 'Year', 'Share'], header=0)
df = pd.merge(left=df_dtp, right=df_measles, on=['Entity', 'Code', 'Year'])
df.dropna(inplace=True)
df = df[df['Year'] == 2024]
df['Share_x'] = df['Share_x']/100; df['Share_y'] = df['Share_y']/100
df.to_csv('../data/treated_data/share_vaccination_world_dtp3_measles.csv')

In [19]:
df2016 = pd.read_csv('../data/US_County_Level_Election_Results_08-24-master/2016_US_County_Level_Presidential_Results.csv', index_col=0)
df2020 = pd.read_csv('../data/US_County_Level_Election_Results_08-24-master/2020_US_County_Level_Presidential_Results.csv')
df2024 = pd.read_csv('../data/US_County_Level_Election_Results_08-24-master/2024_US_County_Level_Presidential_Results.csv')

df2016['county_fips'] = df2016['combined_fips'].astype(int)
df2020['county_fips'] = df2020['county_fips'].astype(int)
df2024['county_fips'] = df2024['county_fips'].astype(int)
    
df2016 = df2016[df2016['county_fips'] % 1000 != 0] 
df2020 = df2020[df2020['county_fips'] % 1000 != 0] 
df2024 = df2024[df2024['county_fips'] % 1000 != 0]

df2016 = df2016[df2016['votes_dem'].fillna(0) + df2016['votes_gop'].fillna(0) > 0]
df2020 = df2020[df2020['votes_dem'].fillna(0) + df2020['votes_gop'].fillna(0) > 0]
df2024 = df2024[df2024['votes_dem'].fillna(0) + df2024['votes_gop'].fillna(0) > 0]

df2016 = df2016[df2016['county_name'] != 'Alaska']

df2020 = df2020.groupby('county_fips', as_index=False)[['votes_dem','votes_gop']].sum()
df2024 = df2024.groupby('county_fips', as_index=False)[['votes_dem','votes_gop']].sum()

df_merge = pd.merge(df2020, df2024, on='county_fips', suffixes=('_20','_24'), how='inner')
df_merge['share_dem_20'] = df_merge['votes_dem_20']/(df_merge['votes_dem_20'] + df_merge['votes_gop_20'])
df_merge['share_dem_24'] = df_merge['votes_dem_24']/(df_merge['votes_dem_24'] + df_merge['votes_gop_24'])
df_merge.drop(columns=['votes_dem_20', 'votes_gop_20', 'votes_dem_24', 'votes_gop_24'], inplace=True)
df_merge.set_index('county_fips', inplace=True)

df_merge.to_csv('../data/treated_data/us_politics.csv')

In [21]:
rng = np.random.default_rng(seed=4721)
U = rng.dirichlet(alpha=(2,7,1,3), size=1000)
x = U[:,0] + U[:,1]
y = U[:,0] + U[:,2]
salvar_xy_csv(x, y, nome_arquivo='../data/treated_data/simulated_bivariate_beta_normal_case.csv')

U = rng.dirichlet(alpha=(0.05,10,1,2), size=1000)
x = U[:,0] + U[:,1]
y = U[:,0] + U[:,2]
salvar_xy_csv(x, y, nome_arquivo='../data/treated_data/simulated_bivariate_beta_complicated_case.csv')